In [43]:
from config import *
import time
import Minimal_Model_1D as mm1D
import Minimal_Model_2D as mm2D
import Operators2D as O2d

In [33]:
def Correlation_2D(A, ax, ay, i, ip, j, jp, dl=1):

    """
        Returns correlation function between point labelled by i, j on a n X m lattice
        
        Args:
            A (float) : normalizing ocnstant for correlation function
            i, j (int) : index of two poins between which correlation function needs to be calculated
            dl (float) : decay length of correlation function 
        
    """

    #x_dis = ((i%n) - (j%n))*ax   ## x distance between two points on lattice with index i,j labelled in graph way 

    #y_dis = ((i//n) - (j//n))*ay   ## y distance between two points on lattice with index i,j labelled in graph way 

    x_dis = (i -ip)*ax   ## x distance between two points on lattice with index i,j labelled in graph way 

    y_dis = (j -jp)*ay   ## y distance between two points on lattice with index i,j labelled in graph way 

    dis_sq = x_dis**2 + y_dis**2

    lmbda_sq = dl**2

    return A**2 * math.exp(-dis_sq/lmbda_sq)

def QW_wf_2d(n, m, i, ax, ay):

    V = np.empty((m*n))

    for i in range(m):

        for j in range(n):

            #V[i*n+j] = math.sqrt(2/(n-1))*math.sqrt(2/(m-1))*math.sin(j*math.pi/(n-1))*math.sin(i*math.pi/(m-1))

            V[i*n+j] = math.sin(j*math.pi/(n))*math.sin(i*math.pi/(m))*math.sqrt(2/(n)/ax)*math.sqrt(2/(m)/ay)

    return V 

def variance_of_equivalent_uncorrelated_disorder(WV, A, n, m, ax, ay, dl=1):

    """
        Returns equivalent <VV> or variance for uncorrelated disorder in 2D given correlated disorder magnitude A   
        
        Args:
            WV (array) : Array of dim (nXm, 1) representing wavefunction 
            A (float) : normalizing ocnstant for correlation function
            dl (float) : decay length of correlation function 
        
    """
 
    WVb = WV.reshape((4,m,n))           ### reshaping one D wavevector for ease of summing with the correlation function which relies on i,j being latice index

    var_corr = np.empty((4*m*n, 4*m*n))                        ### intializing variance of uncorrelated disorder

    for k in range(4):

        for kp in range(4):

            for i in range(n):

                for j in range(m):

                    for ip in range(n):

                        for jp in range(m):

                            ### summing square of given wavefunction with correlation function to find corr_var

                            var_corr[i+j*n+k*m*n, ip+jp*n+kp*m*n] = abs(WVb[k,j,i])**2*abs(WVb[kp,jp,ip])**2*Correlation_2D(A, ax, ay, i, ip, j, jp, dl)
                            

    ### Please note, take the ax**2*ay**2 (area of single element of integral), 
    ### include it either in normalization of wavefunction, or in variance reimann sum calculation. 
    ### Do not include it at both the places.

    var_corr_sum = np.sum(var_corr)*ax**2*ay**2

    var_uncorr = np.empty((m*n, 4, 4))

    for k in range(4):

            for kp in range(4):

                for j in range(m):

                    for i in range(n):

                        ### summing square of given wavefunction with correlation function to find corr_var

                        var_uncorr[i+j*n, k, kp] = abs(WVb[k,j,i])**2*abs(WVb[kp,j,i])**2

    factor = np.sum(var_uncorr)*ax*ay

    eqv_uncorr_sum = var_corr_sum/ factor

    return eqv_uncorr_sum, var_corr_sum, factor


Matching with analytical

In [36]:
N = 20

AX = 0.5

M = 20

AY = 0.5

D = 2


print('LMBDA_SQ = ', (AX**2+AY**2)*D**2)

DSQRT = math.sqrt((AX**2+AY**2)*D**2)

S = np.tile(QW_wf_2d(N, M, 1, AX, AY),4)/2

print(S.shape)

print(np.sum(np.abs(S)**2))

print(np.sum(np.abs(S)**2)*AX*AY)


t1 = time.time()

print(variance_of_equivalent_uncorrelated_disorder(S,1,N,M,AX,AY,DSQRT))

t2 = time.time()

print(t2-t1)

LMBDA_SQ =  2.0
(1600,)
3.9999999999999996
0.9999999999999999
(5.549449274917808, 0.12486260868565066, 0.0225)
4.070502042770386


In [44]:
### optimized code
### Matching with above def, which is already matched with analytical

from config import *

def QW_wf_2d(n, m, i, ax, ay):

    V = np.empty((m*n))

    for i in range(m):

        for j in range(n):

            #V[i*n+j] = math.sqrt(2/(n-1))*math.sqrt(2/(m-1))*math.sin(j*math.pi/(n-1))*math.sin(i*math.pi/(m-1))

            V[i*n+j] = math.sin(j*math.pi/(n))*math.sin(i*math.pi/(m))*math.sqrt(2/(n)/ax)*math.sqrt(2/(m)/ay)

    return V 

class twoD_Minimal_Model:

    def __init__(self, n, m, ax, ay):
        """
        Initializes a new instance of the twoD_Minimal_Model class.
        
        Args:
            n (int): Number of horizontal lattice points.
            m (int): Number of vertical lattice points.
            ax (float): Horizontal lattice spacing.
            ay (float): Vertical lattice spacing.
        """
        self.n = n
        self.m = m
        self.ax = ax
        self.ay = ay

    def Correlation_2D(self, A, i, ip, j, jp, dl=1):

        """
            Returns correlation function between point labelled by i, j on a n X m lattice
            
            Args:
                A (float) : normalizing ocnstant for correlation function
                i, j (int) : index of two poins between which correlation function needs to be calculated
                dl (float) : decay length of correlation function 
            
        """

        #x_dis = ((i%n) - (j%n))*ax   ## x distance between two points on lattice with index i,j labelled in graph way 

        #y_dis = ((i//n) - (j//n))*ay   ## y distance between two points on lattice with index i,j labelled in graph way 

        x_dis = (i -ip)*self.ax   ## x distance between two points on lattice with index i,j labelled in graph way 

        y_dis = (j -jp)*self.ay   ## y distance between two points on lattice with index i,j labelled in graph way 

        dis_sq = x_dis**2 + y_dis**2

        lmbda_sq = dl**2

        return A**2 * np.exp(-dis_sq/lmbda_sq)
    
    def variance_of_equivalent_uncorrelated_disorder(self, WV, A, dl=1):
        
        """
        Calculate the equivalent variance of uncorrelated disorder.
        
        Args:
            WV (np.ndarray): Wavefunction array of shape (n*m, 1).
            A (float): Normalizing constant for the correlation function.
            dl (float): Decay length of the correlation function.
        
        Returns:
            float: Variance of equivalent uncorrelated disorder.
        """
        WVb = WV.reshape((4, self.m, self.n))
        WVb_squared = np.abs(WVb)**2
        
        # Calculate WVb_outer using np.einsum
        WVb_outer = np.einsum('kij,lmn->kijlmn', WVb_squared, WVb_squared)
        print(WVb_outer.shape)

        # Generate the indices and calculate the correlation function
        j, i, jp, ip = np.meshgrid(np.arange(self.m), np.arange(self.n), np.arange(self.m), np.arange(self.n), indexing='ij')
        corr_function = self.Correlation_2D(A, i, ip, j, jp, dl)
        print(corr_function.shape)

        # Multiply WVb_outer to correlation function
        resulting_matrix = np.einsum('kijlmn, ijmn -> kl', WVb_outer, corr_function)

        #calculate the sum
        var_corr = np.sum(resulting_matrix)*self.ax**2*self.ay**2
        
        # Calculate the sum of squared terms separately for normalization
        norm_factor = np.sum(np.einsum('kij,lij->kl', WVb_squared, WVb_squared))*self.ax*self.ay

        # Compute the equivalent uncorrelated disorder variance
        eqv_uncorr =  var_corr / norm_factor

        return eqv_uncorr, var_corr, norm_factor

N = 20

AX = 0.5

M = 20

AY = 0.5

D = 2

DSQRT = math.sqrt((AX**2+AY**2)*D**2)

S = np.tile(QW_wf_2d(N, M, 1, AX, AY),4)/2

t1 = time.time()
# Example usage
model = twoD_Minimal_Model(N, M, AX, AY)  # Initialize model with example parameters
a,b,c = model.variance_of_equivalent_uncorrelated_disorder(S, 1, dl=DSQRT)
t2 = time.time()
print(a,b,c)
print(t2-t1)


(4, 20, 20, 4, 20, 20)
(20, 20, 20, 20)
5.549449274917809 0.12486260868565066 0.022499999999999996
0.012861013412475586


In [45]:
N = 50

AX = 0.2

M = 50

AY = 0.2

D = 5

print('LMBDA_SQ = ', (AX**2+AY**2)*D**2)

DSQRT = math.sqrt((AX**2+AY**2)*D**2)

S = np.tile(QW_wf_2d(N, M, 1, AX, AY),4)/2

print(S.shape)

print(np.sum(np.abs(S)**2))

print(np.sum(np.abs(S)**2)*AX*AY)


t1 = time.time()

print(variance_of_equivalent_uncorrelated_disorder(S,1,N,M,AX,AY,DSQRT))

t2 = time.time()

print(t2-t1)



LMBDA_SQ =  2.0000000000000004
(10000,)
25.0
1.0
(5.549406136135906, 0.12486163806305785, 0.022499999999999992)
177.5128846168518


In [46]:
t1 = time.time()
# Example usage
model = twoD_Minimal_Model(N, M, AX, AY)  # Initialize model with example parameters
a,b,c = model.variance_of_equivalent_uncorrelated_disorder(S, 1, dl=DSQRT)
t2 = time.time()
print(a,b,c)
print(t2-t1)

(4, 50, 50, 4, 50, 50)
(50, 50, 50, 50)
5.549406136135867 0.1248616380630571 0.022500000000000017
0.43117284774780273


In [58]:
N = 23

AX = 0.2

M = 22

AY = 0.2

D = 5

print('LMBDA_SQ = ', (AX**2+AY**2)*D**2)

DSQRT = math.sqrt((AX**2+AY**2)*D**2)

S = np.tile(QW_wf_2d(N, M, 1, AX, AY),4)/2

print(S.shape)

print(np.sum(np.abs(S)**2))

print(np.sum(np.abs(S)**2)*AX*AY)


t1 = time.time()

print(variance_of_equivalent_uncorrelated_disorder(S,1,N,M,AX,AY,DSQRT))

t2 = time.time()

print(t2-t1)



LMBDA_SQ =  2.0000000000000004
(2024,)
24.999999999999996
1.0
(3.7703317440507154, 0.4191327284641357, 0.1111660079051383)
7.141592264175415


In [59]:
t1 = time.time()
# Example usage
model = twoD_Minimal_Model(N, M, AX, AY)  # Initialize model with example parameters
a,b,c = model.variance_of_equivalent_uncorrelated_disorder(S, 1, dl=DSQRT)
t2 = time.time()
print(a,b,c)
print(t2-t1)

(4, 22, 23, 4, 22, 23)
(22, 23, 22, 23)
3.7703317440507154 0.4191327284641357 0.1111660079051383
0.026254653930664062


In [60]:
N = 10

AX = 1

M = 11

AY = 1

D = 1

DSQRT = math.sqrt((AX**2+AY**2)*D**2)

print('d=', DSQRT**2)

S = np.tile(QW_wf_2d(N, M, 1, AX, AY),4)/2

t1 = time.time()
# Example usage
model = twoD_Minimal_Model(N, M, AX, AY)  # Initialize model with example parameters
a,b,c = model.variance_of_equivalent_uncorrelated_disorder(S, 1, dl=DSQRT)
t2 = time.time()
print(a,b,c)
print(t2-t1)


d= 2.0000000000000004
(4, 11, 10, 4, 11, 10)
(11, 10, 11, 10)
5.607347165559055 0.11469573747734435 0.02045454545454546
0.0020017623901367188


In [57]:
N = 10

AX = 1

M = 11

AY = 1

D = 4

DSQRT = math.sqrt((AX**2+AY**2)*D**2)

print('d=', DSQRT**2)

S = np.tile(QW_wf_2d(N, M, 1, AX, AY),4)/2

t1 = time.time()
# Example usage
model = twoD_Minimal_Model(N, M, AX, AY)  # Initialize model with example parameters
a,b,c = model.variance_of_equivalent_uncorrelated_disorder(S, 1, dl=DSQRT)
t2 = time.time()
print(a,b,c)
print(t2-t1)

d= 32.00000000000001
(4, 11, 10, 4, 11, 10)
(11, 10, 11, 10)
5.607347165559055 0.11469573747734435 0.02045454545454546
0.002002239227294922
